FileNotFoundError: Arquivo não encontrado: c:\Users\joaol\ProjetoTCS-2026\Notebook_limpeza_dados\Datasets\lampadas-de-vapor-de-sodio-de-alta-pressao.csv

In [4]:

from pathlib import Path
import pandas as pd
import numpy as np

# Caminho correto: voltar uma pasta e entrar em Datasets
diretorio_atual = Path.cwd()
caminho_final = (
    diretorio_atual.parent
    / "Datasets"
    / "lampadas-de-vapor-de-sodio-de-alta-pressao.csv"
)

if not caminho_final.is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_final}")

df = pd.read_csv(caminho_final, encoding="utf-8-sig")

print("Arquivo carregado com sucesso!")
print("Dimensões:", df.shape)


Arquivo carregado com sucesso!
Dimensões: (763, 12)


In [9]:

from pathlib import Path
import pandas as pd
import numpy as np

# 1. Localizar o arquivo usando caminhos específicos
diretorio_atual = Path.cwd()

arquivo_nome = "lampadas-de-vapor-de-sodio-de-alta-pressao.csv"

caminhos_possiveis = [
    diretorio_atual.parent / "Datasets" / arquivo_nome,
    diretorio_atual / "Datasets" / arquivo_nome,
    diretorio_atual / arquivo_nome
]

caminho_final = next(
    (caminho for caminho in caminhos_possiveis if caminho.is_file()),
    None
)

if caminho_final is None:
    raise FileNotFoundError(
        "Arquivo não encontrado. Verifique se ele está na pasta Datasets."
    )

# 2. Ler o arquivo
df = pd.read_csv(caminho_final, encoding="utf-8-sig")

print("Arquivo carregado:", caminho_final.name)
print("Dados originais:", df.shape)

# 3. Remover colunas administrativas
colunas_remover = [
    "Status do Modelo",
    "Código de Barras",
    "Status do Registro",
    "Número do Registro"
]

df = df.drop(columns=colunas_remover, errors="ignore")

# 4. Remover registros sem marca ou modelo
df = df.dropna(subset=["Marca", "Modelo"])

# 5. Remover duplicatas exatas
df = df.drop_duplicates()

# 6. Converter colunas numéricas
colunas_numericas = [
    "Tensão (V)",
    "Potência (W)",
    "Fluxo Luminoso (lm)",
    "Vida Mediana(horas)",
    "Eficiência Luminosa(lm/W)"
]

for coluna in colunas_numericas:
    if coluna in df.columns:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

# 7. Manter apenas registros com potência válida
df = df[
    df["Potência (W)"].notna()
    & (df["Potência (W)"] > 0)
].copy()

# 8. Tratar valores inválidos nas outras colunas numéricas
for coluna in colunas_numericas:
    if coluna != "Potência (W)" and coluna in df.columns:
        df.loc[df[coluna] <= 0, coluna] = np.nan

# 9. Manter o registro mais completo de cada marca e modelo
df["_completude"] = df.notna().sum(axis=1)

df = (
    df.sort_values("_completude", ascending=False)
    .drop_duplicates(subset=["Marca", "Modelo"])
    .drop(columns=["_completude"])
)

# 10. Salvar o arquivo limpo na pasta do notebook
arquivo_saida = diretorio_atual / "lampadas_vapor_sodio_limpo.csv"

df.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8-sig"
)

print("\nLimpeza concluída!")
print("Dados finais:", df.shape)
print("Arquivo salvo em:", arquivo_saida)
print("\nColunas mantidas:")
print(df.columns.tolist())


Arquivo carregado: lampadas-de-vapor-de-sodio-de-alta-pressao.csv
Dados originais: (763, 12)

Limpeza concluída!
Dados finais: (155, 8)
Arquivo salvo em: c:\Users\joaol\ProjetoTCS-2026\Notebook_limpeza_dados\lampadas_vapor_sodio_limpo.csv

Colunas mantidas:
['Marca', 'Modelo', 'Tensão (V)', 'Potência (W)', 'Fluxo Luminoso (lm)', 'Vida Mediana(horas)', 'Eficiência Luminosa(lm/W)', 'Selo Procel']
